__PIZZA HUT CHATBOT WITH GRADIO__

In [28]:
pip install langchain langchain-ollama gradio --quiet

Note: you may need to restart the kernel to use updated packages.


In [29]:
import sqlite3
from datetime import datetime

def get_connection():
    return sqlite3.connect("pizzahut.db")

conn = get_connection()
cursor = conn.cursor()

# Create menu table
cursor.execute("""
CREATE TABLE IF NOT EXISTS menu (
    id INTEGER PRIMARY KEY,
    name TEXT,
    size TEXT,
    price REAL,
    description TEXT
)
""")

# Create orders table
cursor.execute("""
CREATE TABLE IF NOT EXISTS orders (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    item_name TEXT,
    size TEXT,
    quantity INTEGER,
    price REAL,
    status TEXT,
    created_at TEXT
)
""")

# Seed menu data
cursor.execute("DELETE FROM menu")

menu_items = [
    (1, "Margherita", "Small", 8.0, "tomato sauce, mozzarella, basil"),
    (2, "Pepperoni", "Large", 18.0, "tomato sauce, mozzarella, pepperoni"),
    (3, "Veggie", "Medium", 12.0, "bell peppers, olives, onions"),
    (4, "BBQ Chicken", "Large", 20.0, "bbq sauce, chicken, red onions")
]

cursor.executemany(
    "INSERT INTO menu VALUES (?, ?, ?, ?, ?)", #insert multiple rows into database
    menu_items
)

conn.commit()
conn.close()

print("Database created and menu seeded.")

Database created and menu seeded.


In [30]:
from langchain.tools import tool

@tool
def get_menu() -> str:
    """Fetches all available items from the Pizza Hut menu."""

    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute("SELECT name, size, price, description FROM menu")
    rows = cursor.fetchall()

    conn.close()

    menu_text = "🍕 Pizza Hut Menu:\n\n"

    for name, size, price, desc in rows:
        menu_text += f"{name} ({size}) - ${price}\n{desc}\n\n"

    return menu_text


@tool
def place_order(item_name: str, size: str, quantity: int) -> str:
    """Places an order for a menu item."""

    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute(
        "SELECT price FROM menu WHERE name=? AND size=?",
        (item_name, size)
    )

    result = cursor.fetchone()

    if not result:
        conn.close()
        return "❌ Item not found in menu."

    price = result[0]
    total_price = price * quantity

    cursor.execute("""
    INSERT INTO orders (item_name, size, quantity, price, status, created_at)
    VALUES (?, ?, ?, ?, ?, ?)
    """,
    (item_name, size, quantity, total_price, "Confirmed", datetime.now().isoformat())
    )

    conn.commit()
    conn.close()

    return f"✅ Order placed: {quantity} {size} {item_name} pizza(s). Total: ${total_price}"


@tool
def get_orders() -> str:
    """Retrieves all current orders."""

    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute(
        "SELECT item_name, size, quantity, price, status FROM orders"
    )

    rows = cursor.fetchall()
    conn.close()

    if not rows:
        return "No orders placed yet."

    text = "🧾 Current Orders:\n\n"

    for item, size, qty, price, status in rows:
        text += f"{qty}x {size} {item} - ${price} ({status})\n"

    return text

In [35]:
SYSTEM_PROMPT = """
You are a Pizza Hut ordering assistant.

Rules:
- Use get_menu to show available items
- Use place_order to place customer orders
- Use get_orders to show what has been ordered
- Always confirm the order details after placing
- Always mention prices when showing menu items
"""

In [36]:
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

llm = ChatOllama(
    model="llama3.2",
    temperature=0.7
)

tools = [get_menu, place_order, get_orders]

tool_map = {t.name: t for t in tools}

llm_with_tools = llm.bind_tools(tools)

In [37]:
def run_agent(user_message, history):

    messages = [SystemMessage(content=SYSTEM_PROMPT)]

    # Convert history
    for h in history:
        if h["role"] == "user":
            messages.append(HumanMessage(content=h["content"]))
        elif h["role"] == "assistant":
            messages.append(AIMessage(content=h["content"]))

    messages.append(HumanMessage(content=user_message))

    while True:

        response = llm_with_tools.invoke(messages)

        messages.append(response)

        if not response.tool_calls:
            return response.content

        for tool_call in response.tool_calls:

            tool_name = tool_call["name"]
            tool_args = tool_call["args"]

            tool = tool_map[tool_name]

            result = tool.invoke(tool_args)

            messages.append(
                ToolMessage(
                    content=str(result),
                    tool_call_id=tool_call["id"]
                )
            )

In [15]:
history = []

queries = [
    "What's on the menu?",
    "I'd like to order 2 large Pepperoni pizzas",
    "Show me my orders"
]

for q in queries:
    answer = run_agent(q, history)
    print("User:", q)
    print("Bot:", answer)
    print()

    history.append({"role":"user","content":q})
    history.append({"role":"assistant","content":answer})

User: What's on the menu?
Bot: We have a variety of pizzas on the menu. Would you like to place an order?

User: I'd like to order 2 large Pepperoni pizzas
Bot: It looks like we don't have a Pepperoni Pizza on our menu. Let me show you our menu options instead.

Here are some of our popular pizzas:

1. Meat Lover's Pizza ($16.99) - a hearty combination of pepperoni, sausage, bacon, and ham
2. Veggie Pizza ($14.99) - a flavorful blend of vegetables, including mushrooms, onions, and bell peppers
3. Hawaiian Pizza ($15.99) - a classic combination of ham and pineapple

Would you like to order one of these options instead?

User: Show me my orders
Bot: Let's go back to the menu. Would you like to order one of our pizzas?



In [38]:
import gradio as gr

history = []
chat_history = []

def chat(user_message, chat_history):
    response = run_agent(user_message, history)

    history.append({"role": "user", "content": user_message})
    history.append({"role": "assistant", "content": response})

    chat_history.append({"role": "user", "content": user_message})
    chat_history.append({"role": "assistant", "content": response})

    return "", chat_history

def clear_chat():
    global history, chat_history
    history = []
    chat_history = []
    return []

with gr.Blocks(title="Pizza Hut Order Bot") as demo:
    gr.Markdown("# 🍕 Pizza Hut Order Bot")

    chatbot = gr.Chatbot()
    msg = gr.Textbox(placeholder="Ask for menu or place an order...")
    clear = gr.Button("Clear Chat")

    msg.submit(chat, [msg, chatbot], [msg, chatbot])
    clear.click(clear_chat, None, chatbot)

demo.launch(theme=gr.themes.Soft())

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.
